Technology-wise Cost & Effort Gold

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum, count, round, when

# Initialize SparkSession (if not already)
spark = SparkSession.builder.getOrCreate()

# 1️⃣ Read data from Silver layer
df_silver = spark.table("cockpit.finance.silver_invoice")

# 2️⃣ Compute total Amount globally for SLA % calculation
total_amount = df_silver.agg(sum("Amount_INR").alias("total_amount")).collect()[0]["total_amount"]

# 3️⃣ SLA Performance Gold aggregation per Technology
df_gold_sla = (
    df_silver
    .groupBy("Technology")  # You can also group by "Description" if per activity
    .agg(
        count("*").alias("total_tasks"),
        sum("Hours").alias("total_hours"),
        sum("SLA_Breached").alias("sla_breached_count"),
        sum(when(col("SLA_Breached") == 1, col("Amount_INR"))).alias("sla_breached_cost")
    )
    .withColumn(
        "sla_compliance_pct",
        round( ( (col("total_tasks") - col("sla_breached_count")) / col("total_tasks") ) * 100, 2)
    )
    .withColumn(
        "sla_breached_cost_pct",
        round( (col("sla_breached_cost") / total_amount) * 100, 2)
    )
)

# 4️⃣ Save the SLA Gold table in Delta
df_gold_sla.write.format("delta").mode("overwrite") \
.saveAsTable("cockpit.finance.gold_sla_performance")

# 5️⃣ Optional: Show result
df_gold_sla.show(truncate=False)


In [0]:
display(df_gold)